# NEAT Inference Notebook: Constrained Strategy Assignment

This notebook loads the trained NEAT winner genome and performs constrained inference over the held-out test split.

## Objective
Assign an AI strategy to each test record while respecting per-day plant capacity constraints.

## Workflow
1. Resolve project paths and load artifacts
2. Build NEAT network from winner genome
3. Encode each record into the 9-feature input vector
4. Apply capacity-aware strategy selection
5. Summarize AI strategy distribution

The inference dataset is the reserved test split so the output can be used as a true generalization check.

In [1]:
import pickle
import time
from pathlib import Path

import joblib
import neat
import pandas as pd

## 1. Resolve Paths and Validate Inputs

Find the project root robustly from notebook-relative candidates and verify all required files exist before inference starts.

In [2]:
def resolve_project_root(candidates):
    """Return the first folder that looks like the project root."""
    for candidate in candidates:
        c = candidate.resolve()
        if (c / "config").exists() and (c / "data").exists() and (c / "models").exists():
            return c
    raise FileNotFoundError(f"No valid project root found among: {candidates}")


project_root_candidates = [
    Path(".."),
    Path("../.."),
    Path("."),
]
project_root = resolve_project_root(project_root_candidates)

config_path = project_root / "config" / "config-feedforward.txt"
model_path = project_root / "models" / "artifacts" / "winner_genome.pkl"
test_split_path = project_root / "data" / "split" / "dataset_optimization_cereal_co2_test_raw.csv"
scaler_path = project_root / "data" / "split" / "dataset_optimization_cereal_co2_scaler.joblib"

if not config_path.exists():
    raise FileNotFoundError(f"NEAT config does not exist: {config_path}")
if not model_path.exists():
    raise FileNotFoundError(f"Winner genome pickle does not exist: {model_path}")
if not test_split_path.exists():
    raise FileNotFoundError(f"Inference test split does not exist: {test_split_path}")
if not scaler_path.exists():
    raise FileNotFoundError(f"Preprocessing scaler does not exist: {scaler_path}")

print(f"Project root: {project_root}")
print(f"Config path: {config_path}")
print(f"Model path: {model_path}")
print(f"Test split path: {test_split_path}")
print(f"Scaler path: {scaler_path}")

Project root: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos
Config path: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\config\config-feedforward.txt
Model path: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\models\artifacts\winner_genome.pkl
Test split path: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_test_raw.csv
Scaler path: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\split\dataset_optimization_cereal_co2_scaler.joblib


## 2. Load Winner Genome and Build NEAT Network

Load the trained winner genome and instantiate the feed-forward NEAT network using the same configuration file used during training.

In [3]:
print(f"Loading winner genome from: {model_path}")
with model_path.open("rb") as file_obj:
    winner_genome = pickle.load(file_obj)

print(f"Loading NEAT config from: {config_path}")
neat_config = neat.Config(
    neat.DefaultGenome,
    neat.DefaultReproduction,
    neat.DefaultSpeciesSet,
    neat.DefaultStagnation,
    str(config_path),
)

winner_network = neat.nn.FeedForwardNetwork.create(winner_genome, neat_config)
print("Winner network ready.")

Loading winner genome from: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\models\artifacts\winner_genome.pkl
Loading NEAT config from: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\config\config-feedforward.txt
Winner network ready.


## 3. Capacity-Constrained Inference

For each record in the held-out test split:
- Build the same 9-feature input vector used in training
- Score strategies with the winner network
- Keep only capacity-feasible strategies
- Choose the feasible strategy with minimum estimated emissions
- Break ties by neural score

Capacities reset every `LOTS_PER_DAY` records.

In [4]:
df_inference = pd.read_csv(test_split_path)

# Load preprocessing scaler artifact and recover physical min/max used during training
scaler = joblib.load(scaler_path)
PHYSICAL_COLUMNS = ["generated_volume_tons", "moisture_pct", "process_temperature_c"]
physical_bounds = {
    column: (float(min_value), float(max_value))
    for column, min_value, max_value in zip(PHYSICAL_COLUMNS, scaler.data_min_, scaler.data_max_)
}

LOTS_PER_DAY = 15
FALLBACK_STRATEGY = "Biomass combustion"
available_strategies = [
    "Biomass combustion",
    "Animal feed",
    "Composting",
    "Biochar",
]

def normalize_minmax(value_real: float, min_value: float, max_value: float) -> float:
    denominator = max_value - min_value
    if denominator <= 0:
        raise ValueError(f"Invalid normalization bounds: min={min_value}, max={max_value}")
    value_norm = (value_real - min_value) / denominator
    return max(0.0, min(1.0, float(value_norm)))

def neural_preferences(row) -> dict[str, float]:
    vol_min, vol_max = physical_bounds["generated_volume_tons"]
    hum_min, hum_max = physical_bounds["moisture_pct"]
    temp_min, temp_max = physical_bounds["process_temperature_c"]

    vol_norm = normalize_minmax(row.generated_volume_tons, vol_min, vol_max)
    hum_norm = normalize_minmax(row.moisture_pct, hum_min, hum_max)

    temp_norm = 0.0
    if hasattr(row, "process_temperature_c") and row.process_temperature_c is not None:
        temp_norm = normalize_minmax(row.process_temperature_c, temp_min, temp_max)

    is_rainy = 1 if row.season == "Rainy" else 0
    is_dry = 1 if row.season == "Dry" else 0
    is_husk = 1 if row.subproduct_type == "Husk" else 0
    is_straw = 1 if row.subproduct_type == "Straw" else 0
    is_silo_dust = 1 if row.subproduct_type == "Silo dust" else 0
    is_bran = 1 if row.subproduct_type == "Bran" else 0

    inputs = [
        vol_norm,
        hum_norm,
        temp_norm,
        is_husk,
        is_straw,
        is_silo_dust,
        is_bran,
        is_rainy,
        is_dry,
    ]

    output = winner_network.activate(inputs)
    return {
        "Biomass combustion": float(output[0]),
        "Animal feed": float(output[1]),
        "Composting": float(output[2]),
        "Biochar": float(output[3]),
    }

def estimated_emissions_for_strategy(row, strategy: str) -> float:
    temperature_c = row.process_temperature_c if hasattr(row, "process_temperature_c") else 180.0
    emission_base = (temperature_c * 0.5) * row.generated_volume_tons

    if strategy == "Biomass combustion":
        humidity_term = (row.moisture_pct ** 1.5) * 2.0
        return emission_base + humidity_term - 50.0
    if strategy == "Animal feed":
        if row.subproduct_type in {"Husk", "Straw", "Silo dust"} or row.moisture_pct > 18.0:
            return emission_base * 1.8
        return emission_base * 0.4
    if strategy == "Biochar":
        if row.moisture_pct < 10.0:
            return emission_base * 0.2
        return emission_base * 1.2
    if strategy == "Composting":
        return emission_base * 0.8 + (row.generated_volume_tons * 5.0)
    return emission_base

def choose_strategy(row, current_capacities: dict[str, float]) -> str:
    ranking = sorted(
        neural_preferences(row).items(),
        key=lambda item: item[1],
        reverse=True,
    )

    feasible_candidates = []
    for strategy, score in ranking:
        if row.generated_volume_tons > current_capacities.get(strategy, 0.0):
            continue
        emissions = estimated_emissions_for_strategy(row, strategy)
        feasible_candidates.append((emissions, -score, strategy))

    if feasible_candidates:
        _, _, chosen = min(feasible_candidates)
        current_capacities[chosen] -= row.generated_volume_tons
        return chosen

    return FALLBACK_STRATEGY

print(f"Starting inference for {len(df_inference)} test records...")
start_time = time.time()

ai_assignments = []
current_capacities = {}

for index, row in enumerate(df_inference.itertuples(index=False)):
    if index % LOTS_PER_DAY == 0:
        current_capacities = {
            "Animal feed": 50.0,
            "Composting": 100.0,
            "Biochar": 15.0,
            "Biomass combustion": 5000.0,
        }

    try:
        ai_assignments.append(choose_strategy(row=row, current_capacities=current_capacities))
    except Exception as exc:
        print(f"Row {index} failed during inference: {exc}")
        ai_assignments.append(FALLBACK_STRATEGY)

df_inference["ai_assigned_strategy"] = ai_assignments

Starting inference for 10000 test records...


## 4. Results and Export

Report runtime and strategy distribution, then save the inference output for downstream evaluation.

In [5]:
end_time = time.time()
print(f"Inference completed in {round(end_time - start_time, 2)} seconds.\n")

print("--- AI Strategy Distribution (Capacity-Constrained) ---")
print(df_inference["ai_assigned_strategy"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

output_path = project_root / "data" / "predictions" / "inference_with_constraints.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df_inference.to_csv(output_path, index=False)
print(f"\nInference dataset saved to: {output_path}")

Inference completed in 0.11 seconds.

--- AI Strategy Distribution (Capacity-Constrained) ---
ai_assigned_strategy
Biomass combustion    43.44%
Composting            37.26%
Animal feed           10.86%
Biochar                8.44%
Name: proportion, dtype: object

Inference dataset saved to: C:\Users\IA\Desktop\Proyectos\DATAGIA\a33-cnp-cereals-neuroevolutivo-reduccion-ambiental-residuos\data\predictions\inference_with_constraints.csv
